# 🔍 Silver EDA — Exploratory Data Analysis
**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Source:** main.default.silver_warehouse_sales

### Business Questions
1. Which item types generate the most sales volume?
2. Is there seasonality — months or years with higher sales?
3. Who are the top suppliers by total sales?
4. Which products sell the most on average?
5. Do customers prefer retail or warehouse channel?
6. Which suppliers carry the most product variety?

In [0]:
import sys
from datetime import datetime
from pyspark.sql import functions as F

# Define table name once — change here if catalog/schema changes
SILVER_TABLE = "main.default.silver_warehouse_sales"

# Expected schema validation — fails loudly if Silver table changed
EXPECTED_COLS = [
    'date', 'year', 'month', 'supplier', 'item_code',
    'item_description', 'item_type', 'retail_sales',
    'retail_transfers', 'warehouse_sales', 'total_sales'
]

try:
    df = spark.table(SILVER_TABLE)

    # Schema validation
    missing = set(EXPECTED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in Silver table: {missing}")
    
    # Validación extra de columnas no esperadas
    extra = set(df.columns) - set(EXPECTED_COLS)
    if extra:
        print(f"  ⚠ Extra columns found: {extra}")

    # Get row count for summary
    row_count = df.count()

    print("=" * 50)
    print("  ENVIRONMENT & DATA SUMMARY")
    print("=" * 50)
    print(f"  Spark version  : {spark.version}")
    print(f"  Python version : {sys.version.split()[0]}")
    print(f"  Run timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Table          : {SILVER_TABLE}")
    print(f"  Rows           : {row_count:,}")
    print(f"  Columns        : {len(df.columns)}")
    print(f"  Schema         : validated ✓")
    print(f"  Compute        : Serverless (auto-optimized)")
    print("=" * 50)

except ValueError as ve:
    print(f"  ✗ Schema validation failed: {ve}")
    raise
except Exception as e:
    print(f"  ✗ Failed to load Silver table: {e}")
    raise

In [0]:
# Create clean DataFrame to avoid repeating filters
# Excludes REF, DUNNAGE and UNCLASSIFIED across all analyses
# ℹ️ No explicit cache needed on Serverless — automatic result caching handles reuse
df_clean = df.filter(
    ~F.col("item_type").isin("REF", "DUNNAGE", "UNCLASSIFIED")
)

clean_count = df_clean.count()
print(f"Clean rows (excluding REF/DUNNAGE/UNCLASSIFIED): {clean_count:,}")

%md
## Analysis 1 — Sales Distribution by Item Type
**Question:** Which item types generate the most sales volume?  
**Why it matters:** Understanding category dominance helps prioritize 
inventory, forecasting, and Gold layer business metrics.

> **Filtering note:** `REF`, `DUNNAGE`, and `UNCLASSIFIED` are excluded from 
> all analyses. `REF` and `DUNNAGE` represent returns and packaging adjustments 
> (negative values — not data errors). `UNCLASSIFIED` is the single null row 
> filled during Silver transformation. None represent real sales activity.

In [0]:
# Question 1: Which item types generate the most total sales?
# Using df_clean (already excludes REF, DUNNAGE and UNCLASSIFIED)

total_units = df_clean.agg(F.sum("total_sales")).collect()[0][0]  # total global para calcular %

sales_by_type = df_clean \
    .groupBy("item_type") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales_per_transaction"),
        F.count("*").alias("transaction_count")
    ) \
    .withColumn(
        "pct_of_total",
        F.round((F.col("total_sales") / total_units) * 100, 1)
    ) \
    .orderBy(F.col("total_sales").desc())

display(sales_by_type)

Databricks visualization. Run in Databricks to view.

%md
### Key Findings — Item Type Distribution

> **Unit clarification:** Values represent **units moved**, not dollars,
> per the official dataset description.

| Item Type | Total Units | % of Total | Avg Units/Transaction | Transactions |
|-----------|-------------|------------|----------------------|--------------|
| BEER | 7,668,171.04 | 62.8% | 180.80 | 42,413 |
| WINE | 2,638,101.54 | 21.6% | 14.06 | 187,640 |
| LIQUOR | 1,692,333.41 | 13.9% | 26.07 | 64,910 |
| KEGS | 118,430.00 | 1.0% | 11.67 | 10,146 |
| NON-ALCOHOL | 86,900.28 | 0.7% | 45.55 | 1,908 |
| STR_SUPPLIES | 13,587.46 | 0.1% | 33.55 | 405 |

**Insights:**
- **BEER dominates volume** (62.8% of all units) with the highest avg per transaction (180.8 units) — consistent with bulk warehouse distribution patterns.
- **WINE dominates frequency** (187,640 transactions = 62% of all transactions) but moves in small quantities (avg 14 units) — retail-driven, high-turnover pattern.
- **LIQUOR** is balanced: mid-volume, mid-frequency — likely a mix of retail and wholesale.
- **KEGS + NON-ALCOHOL + STR_SUPPLIES** combined represent less than 2% of total volume — candidates for exclusion or aggregation in Gold layer metrics.

## Analysis 2 — Sales Trends Over Time
**Question:** How have total sales evolved by year and month?  
**Why it matters:** Identifying seasonality and growth trends is essential
for forecasting and ML feature engineering.

In [0]:
# Question 2: How do sales trend over time?
# We group by year and month to see monthly totals
# This reveals seasonality patterns and year-over-year growth

monthly_sales = df_clean.groupBy("year", "month") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.sum("retail_sales"), 2).alias("retail_sales"),
        F.round(F.sum("warehouse_sales"), 2).alias("warehouse_sales"),
        F.count("*").alias("transaction_count")
    ) \
    .orderBy("year", "month")

display(monthly_sales)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Key Findings — Monthly Sales Trends

| Year | Months Available | Coverage | Total Units |
|------|-----------------|----------|-------------|
| 2017 | Jun–Dec | 7 of 12 months | 3,744,155.70 |
| 2018 | Jan–Feb | 2 of 12 months | 837,249.83 |
| 2019 | Jan–Nov | 11 of 12 months | 5,530,405.78 |
| 2020 | Jan, Mar, Jul, Sep | 4 of 12 months | 2,105,712.42 |

- **2019 is the only reliable year** — 11 consecutive months, highest total volume (5.53M units)
- **2018 is nearly unusable** — only Jan–Feb available (837K units, 15% of 2019)
- **2020 data is sparse and irregular** — 4 non-consecutive months; gaps likely related to COVID-19, cause unconfirmed
- **Peak month: Jul 2020** — 597,753 units; **highest 2019 month: May** — 572,351 units
- **Jun 2017 is the highest single month overall** — 577,838 units, despite being the dataset's first month
- **Warehouse sales consistently dominate retail** — in 2019, warehouse was 3.8x retail (3.61M vs 0.96M units)

> ⚠️ **ML implication:** Dataset is not continuous. Missing months are
> absent from the source — not zero sales. Year cannot be used directly
> as a feature without careful handling. Consider focusing ML modeling
> on 2019 data only for time-series tasks that require continuity.

## Analysis 3 — Top Suppliers by Total Sales
**Question:** Which suppliers generate the most sales volume?  
**Why it matters:** Identifying key suppliers helps understand 
market concentration and dependency risks.

In [0]:
# Question 3: Which suppliers generate the most total sales?
# We exclude UNKNOWN suppliers (rows where supplier was null in Bronze)
# top 15 gives enough coverage without overwhelming the visualization

top_suppliers = df_clean.filter(F.col("supplier") != "UNKNOWN") \
    .groupBy("supplier") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales_per_transaction"),
        F.count("*").alias("transaction_count"),
        F.countDistinct("item_type").alias("item_types_carried")
    ) \
    .orderBy(F.col("total_sales").desc()) \
    .limit(15)

display(top_suppliers)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

%md
### Key Findings — Top 15 Suppliers by Total Sales

> **Filtering note:** `UNKNOWN` supplier excluded — represents rows where
> supplier was null in Bronze, filled during Silver transformation.

| Supplier | Total Units | % of Top 15 | Avg Units/Transaction | Item Types | Profile |
|----------|-------------|-------------|----------------------|------------|---------|
| CROWN IMPORTS | 1,819,141.93 | 20.7% | 1,440.33 | 2 | Bulk mover |
| ANHEUSER BUSCH INC | 1,616,887.68 | 18.4% | 279.84 | 3 | High volume |
| MILLER BREWING COMPANY | 1,597,808.94 | 18.1% | 493.46 | 3 | High volume |
| HEINEKEN USA | 940,796.79 | 10.7% | 737.30 | 3 | Bulk mover |
| E & J GALLO WINERY | 528,656.13 | 6.0% | 48.88 | 3 | High frequency |
| DIAGEO NORTH AMERICA INC | 459,921.54 | 5.2% | 61.16 | 5 | Diversified |
| CONSTELLATION BRANDS | 380,826.10 | 4.3% | 56.98 | 2 | Mid-tier |
| BOSTON BEER CORPORATION | 271,825.23 | 3.1% | 167.90 | 3 | Mid-tier |
| THE WINE GROUP | 200,316.64 | 2.3% | 49.62 | 1 | Specialized |
| JIM BEAM BRANDS CO | 199,336.22 | 2.3% | 33.26 | 2 | High frequency |
| FLYING DOG BREWERY LLLP | 180,652.42 | 2.1% | 150.29 | 2 | Mid-tier |
| YUENGLING BREWERY | 179,577.69 | 2.0% | 391.24 | 2 | Bulk mover |
| SAZERAC CO | 156,314.28 | 1.8% | 35.58 | 3 | High frequency |
| BACARDI USA INC | 142,401.43 | 1.6% | 42.28 | 2 | Mid-tier |
| PABST BREWING CO | 134,083.93 | 1.5% | 199.23 | 2 | Mid-tier |

**Insights:**
- **Market is highly concentrated** — top 3 suppliers (Crown, AB InBev, Miller) represent 57.1% of top-15 volume. High dependency risk.
- **Crown Imports stands out** — highest avg per transaction (1,440 units) with only 1,263 transactions. Pure bulk B2B distribution, very few but massive orders.
- **Beer dominates the top 4** — Crown, Anheuser Busch, Miller, Heineken are all beer suppliers. Consistent with Analysis 1 findings.
- **Two distinct supplier profiles emerge:**
  - **Bulk movers** (Crown, Heineken, Yuengling): high avg/transaction, fewer transactions — warehouse-driven
  - **High frequency** (Gallo, Diageo, Jim Beam, Sazerac): low avg/transaction, many transactions — retail-driven
- **DIAGEO is the most diversified** — only supplier carrying 5 item types across the top 15.
- **THE WINE GROUP** carries only 1 item type — most specialized supplier in the top 15.

> ⚠️ **ML implication:** Supplier identity is a high-cardinality categorical feature.
> Direct one-hot encoding is not viable. Consider target encoding or grouping by
> supplier profile (bulk vs frequency) as an engineered feature.

%md
## Analysis 3.1 — Supplier Market Concentration

**Question:** How dependent is the market on its top 3 suppliers?  
**Why it matters:** High concentration in a few suppliers represents a supply chain risk — if any one of them reduces volume, the entire market is impacted.

In [0]:
# Calculate exact market concentration of top 3 suppliers
# This quantifies the supplier dependency risk mentioned in the markdown

# ✅ Using first() instead of collect()[0] — more efficient for single-row results
total_market_row = df_clean.agg(
    F.sum("total_sales").alias("total_market_sales")
).first()

total_market = total_market_row["total_market_sales"] or 0

top3_sales_row = df_clean.filter(
    F.col("supplier").isin(
        "CROWN IMPORTS",
        "MILLER BREWING COMPANY",
        "ANHEUSER BUSCH INC"
    )
).agg(F.sum("total_sales").alias("top3_sales")).first()

top3_sales = top3_sales_row["top3_sales"] or 0

concentration_pct = round(top3_sales / total_market * 100, 1) if total_market > 0 else 0

print("=" * 50)
print("  SUPPLIER CONCENTRATION ANALYSIS")
print("=" * 50)
print(f"  Total market volume : {total_market:,.0f} units")
print(f"  Top 3 suppliers     : {top3_sales:,.0f} units")
print(f"  Market concentration: {concentration_pct}%")
print("=" * 50)

total_suppliers = df_clean.filter(F.col("supplier") != "UNKNOWN").select("supplier").distinct().count()
crown_pct = round(1819141.93 / 12217524 * 100, 1)
print(f"Total distinct suppliers: {total_suppliers}")
print(f"Crown Imports % of total market: {crown_pct}%")

### Key Findings — Supplier Concentration

| Metric | Value |
|--------|-------|
| Total distinct suppliers | 395 |
| Total market volume | 12,217,524 units |
| Top 3 suppliers (Crown, AB InBev, Miller) | 5,033,839 units |
| Market concentration (top 3) | 41.2% |
| Crown Imports alone | 14.9% of total market |

**Insights:**
- **3 suppliers out of 395 control 41.2% of total market volume** — extreme concentration for a market with nearly 400 active suppliers.
- **Crown Imports alone represents 14.9% of the entire market** with only 1,263 transactions — the fewest among the top 15, yet the highest volume. Pure bulk B2B distribution.
- All top 3 are beer distributors — consistent with Analysis 1 (BEER = 62.8% of volume) and Analysis 3 findings.

> ⚠️ **ML implication:** 395 suppliers with extreme volume imbalance makes `supplier` a high-cardinality, skewed feature. Direct one-hot encoding is not viable. Consider grouping by tier (top 3 / top 15 / rest) as an engineered feature instead.

## Analysis 4 — Retail vs Warehouse Sales Channel
**Question:** Do customers prefer retail or warehouse channel?  
**Why it matters:** Understanding channel distribution helps Montgomery 
County optimize inventory allocation between stores and warehouse.

In [0]:
# Question 4: How do retail and warehouse channels compare?
# Using df_clean (already excludes REF, DUNNAGE and UNCLASSIFIED)
# Division by zero protection added — if total_sales = 0, percentage = 0

channel_comparison = df_clean \
    .groupBy("item_type") \
    .agg(
        F.round(F.sum("retail_sales"), 2).alias("total_retail"),
        F.round(F.sum("retail_transfers"), 2).alias("total_transfers"),
        F.round(F.sum("warehouse_sales"), 2).alias("total_warehouse"),
        F.round(F.sum("total_sales"), 2).alias("total_sales")
    ) \
    .withColumn("retail_pct",
        F.when(F.col("total_sales") == 0, 0)
        .otherwise(F.round(F.col("total_retail") / F.col("total_sales") * 100, 1))
    ) \
    .withColumn("transfers_pct",
        F.when(F.col("total_sales") == 0, 0)
        .otherwise(F.round(F.col("total_transfers") / F.col("total_sales") * 100, 1))
    ) \
    .withColumn("warehouse_pct",
        F.when(F.col("total_sales") == 0, 0)
        .otherwise(F.round(F.col("total_warehouse") / F.col("total_sales") * 100, 1))
    ) \
    .orderBy(F.col("total_sales").desc())

display(channel_comparison)

%md
### Key Findings — Sales Channel Analysis

| Item Type | Retail % | Transfers % | Warehouse % |
|-----------|----------|-------------|-------------|
| BEER | 7.5% | 7.4% | 85.1% |
| WINE | 28.3% | 27.8% | 43.9% |
| LIQUOR | 47.4% | 47.0% | 5.6% |
| KEGS | 0.0% | 0.0% | 100.0% |
| NON-ALCOHOL | 39.2% | 30.7% | 30.1% |
| STR_SUPPLIES | 20.2% | 79.8% | 0.0% |

> **Retail Transfers** represent stock moved between county stores —
> not new sales. LIQUOR has the highest transfer rate (47.0%),
> suggesting frequent rebalancing of inventory between retail locations.

**Insights:**
- **BEER is pure bulk B2B** — 85.1% warehouse, consistent with Analysis 3 (Crown, AB InBev, Miller dominating via bulk orders).
- **KEGS are 100% warehouse** — exclusively B2B, no retail presence whatsoever.
- **LIQUOR is primarily retail consumer** — 47.4% retail + 47.0% transfers = 94.4% store-facing. Warehouse almost irrelevant (5.6%).
- **WINE is the most balanced category** — distributed across all three channels with warehouse still dominant (43.9%).
- **STR_SUPPLIES is transfer-dominated** — 79.8% transfers, 0% warehouse. Store supplies move between retail locations only.
- **NON-ALCOHOL** is the only category with roughly equal presence across all three channels (39.2 / 30.7 / 30.1).

> ⚠️ **ML implication:** Each `item_type` has a completely different channel
> pattern. This makes `item_type` a strong categorical feature — consider
> one-hot encoding given only 6 categories (low cardinality, no risk of dimensionality explosion).

## Analysis 5 — Top Products by Sales Volume
**Question:** Which individual products sell the most?  
**Why it matters:** Category-level analysis tells us what type sells most,
but product-level analysis tells us which specific items drive that volume.
This is actionable for inventory and procurement decisions.

In [0]:
# Question 5: Which individual products sell the most?
# Using df_clean (already excludes REF, DUNNAGE and UNCLASSIFIED)
# item_description + item_type + supplier gives full product context

top_products = df_clean \
    .groupBy("item_description", "item_type", "supplier") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.sum("retail_sales"), 2).alias("total_retail"),
        F.round(F.sum("warehouse_sales"), 2).alias("total_warehouse"),
        F.count("*").alias("transaction_count")
    ) \
    .orderBy(F.col("total_sales").desc()) \
    .limit(20)

display(top_products)

### Key Findings — Top 20 Products

**All top 20 products are BEER.** No wine, liquor, or any other category
appears in the top 20 by total volume.

| Rank | Product | Supplier | Total Sales | Warehouse % | Transactions |
|------|---------|----------|-------------|-------------|--------------|
| 1 | CORONA EXTRA LOOSE NR - 12OZ | Crown Imports | 352,574.83 | 86.0% | 24 |
| 2 | CORONA EXTRA 2/12 NR - 12OZ | Crown Imports | 266,992.08 | 92.9% | 24 |
| 3 | HEINEKEN LOOSE NR - 12OZ | Heineken USA | 206,675.17 | 83.2% | 24 |
| 4 | HEINEKEN 2/12 NR - 12OZ | Heineken USA | 169,564.90 | 91.2% | 24 |
| 5 | MILLER LITE 30PK CAN - 12OZ | Miller Brewing | 162,971.40 | 82.5% | 24 |
| 6 | CORONA EXTRA 4/6 NR - 12OZ | Crown Imports | 140,151.32 | 89.3% | 24 |
| 7 | MODELO ESPECIAL 24 LOOSE NR - 12OZ | Crown Imports | 126,634.80 | 92.4% | 24 |
| 8 | BUD LIGHT 30PK CAN | Anheuser Busch | 120,735.97 | 79.8% | 24 |
| 9 | HEINEKEN 4/6 NR - 12OZ | Heineken USA | 110,962.92 | 89.6% | 24 |
| 10 | CORONA EXTRA 18PK NR - 12OZ | Crown Imports | 100,913.44 | 99.4% | 24 |
| 11 | MODELO ESPECIAL 2/12 NR - 12OZ | Crown Imports | 99,779.39 | 95.0% | 24 |
| 12 | MODELO ESPECIAL 2/12 CAN - 12OZ | Crown Imports | 95,371.71 | 95.5% | 24 |
| 13 | COORS LT 30PK CAN - 12OZ | Miller Brewing | 94,733.67 | 83.1% | 24 |
| 14 | MODELO ESPECIAL SUITCASE CANS - 12OZ | Crown Imports | 94,431.88 | 97.0% | 24 |
| 15 | STELLA ARTOIS 2/12 NR - 11.2OZ | Anheuser Busch | 84,581.19 | 81.2% | 24 |
| 16 | BUD 30PK CAN - 12OZ | Anheuser Busch | 79,821.11 | 83.7% | 24 |
| 17 | MILWAUKEE BEST ICE 2/15PK CAN - 12OZ | Miller Brewing | 75,788.35 | 97.2% | 15 |
| 18 | MILLER LITE 18PK CAN - 12OZ | Miller Brewing | 75,072.38 | 97.0% | 24 |
| 19 | MODELO ESPECIAL 18PK CAN - 12OZ | Crown Imports | 74,507.10 | 99.9% | 24 |
| 20 | MILLER LITE 2/12 CAN - 12OZ | Miller Brewing | 68,580.71 | 97.7% | 24 |

**Supplier distribution in top 20:**
| Supplier | Products in Top 20 |
|----------|--------------------|
| Crown Imports | 9 |
| Miller Brewing Company | 5 |
| Heineken USA | 3 |
| Anheuser Busch Inc | 3 |

**Insights:**
- **Crown Imports dominates** — 9 of the top 20 products are Corona or Modelo variants. Single-supplier dependency is extreme at the product level.
- **19 of 20 products have exactly 24 transactions** — sold every single month of the dataset. Only Milwaukee Best Ice has 15, suggesting it was discontinued or added mid-dataset.
- **Warehouse channel dominates across all top 20** — ranging from 79.8% (Bud Light) to 99.9% (Modelo Especial 18pk). These products move almost exclusively through B2B bulk orders.
- **Corona Extra alone (rank 1) represents 352,574 units** — 4.6% of all BEER volume from a single SKU.

> ⚠️ **ML implication:** `item_description` is extremely high cardinality. However, transaction_count = 24 on top products confirms temporal consistency — these SKUs are reliable for time-series modeling. Consider creating a `is_top_product` binary feature rather than encoding all SKUs directly.

%md
---

## EDA Summary — Key Takeaways & Next Steps

### 📊 Dataset Overview

| Metric | Value |
|--------|-------|
| Total records | 307,645 |
| Date range | Jun 2017 – Sep 2020 |
| Best year for modeling | 2019 — 11 consecutive months |
| Distinct suppliers | 395 |
| Clean item types | 6 |

---

### 🔑 5 Things We Learned

**1) This is a beer market.**
BEER represents 62.8% of all volume moved. The top 20 products are 100% beer.
If you could only model one category, it would be BEER.

**2) 3 suppliers control almost half the market.**
Crown Imports, Anheuser Busch, and Miller account for 41.2% of total volume (5,033,839 units).
Crown alone has 9 products and moves 14.9% of the entire market. High dependency, high risk.

**3) How a product sells depends entirely on what it is.**
- 🍺 BEER & KEGS → 85–100% warehouse (bulk B2B orders)
- 🥃 LIQUOR → 94% store-facing (direct to consumer)
- 🍷 WINE → balanced across all three channels

There is no "average" behavior in this dataset. Each category is its own market.

**4) 2018 and 2020 data are incomplete.**
2018 has only 2 months. 2020 has 4 non-consecutive months.
Only 2019 is reliable for trend analysis or time-series modeling.

**5) Top products are rock-solid stable.**
19 of the top 20 products appear in all 24 months of the dataset without exception.
Predictable, consistent, and ideal for forecasting.

---

### ⚙️ Feature Engineering Priorities

| Feature | How to use it | Why |
|---------|--------------|-----|
| `item_type` | One-hot encode | Only 6 values — strongest signal in the dataset |
| `year` / `month` | Keep as-is | Already extracted — capture seasonality directly |
| `supplier` | Group into tiers (top 3 / top 15 / rest) | 395 unique values — too high cardinality to encode directly |
| `is_top_product` | Binary flag | Top 19 SKUs are temporally consistent — strong forecasting signal |
| `item_description` | Exclude or hash | 4,000+ unique SKUs — direct encoding not viable |

---

### 🚀 Recommended Next Steps

**Gold layer**
- Filter to 2019 for any time-series analysis
- Exclude STR_SUPPLIES and KEGS from business KPIs (combined < 1.1% of volume)
- Aggregate by `item_type` + `supplier_tier` + `month` for clean business metrics

**ML layer**
- Focus initial models on BEER — 62.8% of data, most signal
- Use 2019 as training base — only year with near-complete temporal coverage
- Target variable candidates: `total_sales`, `warehouse_sales`, or demand tier classification

---
*Notebook: `03_silver_EDA` — Santiago López Blanco — May 2026*